# 第 16 章互动实验：对齐、亚像素与周期图案

所有估计都调用你在 `16-1_相位相关与亚像素.py` 里的实现。先预测结果，再运行。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "16_图像对齐与亚像素"
ASSIGNMENT = "16-1_相位相关与亚像素.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
def texture(shape=(96, 96), seed=0, width=0.08):
    rng = np.random.default_rng(seed)
    f = np.fft.fft2(rng.normal(size=shape))
    fy = np.fft.fftfreq(shape[0])[:, None]; fx = np.fft.fftfreq(shape[1])[None, :]
    return np.real(np.fft.ifft2(f * np.exp(-(fy**2 + fx**2) / (2 * width**2))))

def fourier_shift(img, dy, dx):
    fy = np.fft.fftfreq(img.shape[0])[:, None]; fx = np.fft.fftfreq(img.shape[1])[None, :]
    return np.real(np.fft.ifft2(np.fft.fft2(img) * np.exp(-2j * np.pi * (fy * dy + fx * dx))))

ref = texture()
y, x = np.mgrid[0:96, 0:96]
periodic = ((x % 8) < 4).astype(float) + 0.5 * (((y % 8) < 4))
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(ref, cmap="gray"); ax[0].set_title("随机纹理")
ax[1].imshow(periodic, cmap="gray"); ax[1].set_title("周期 8 px 的图案")
plt.show()

## 实验 1：亚像素误差随小数位移的变化

预测：误差会不会随小数部分呈周期性变化？最大误差在 0.25 还是 0.5 附近？

In [ ]:
fracs = np.linspace(0, 1, 21)
errs = []
for f in fracs:
    est = attempt(mod.phase_correlation, ref, fourier_shift(ref, 3 + f, -2 - f), use_window=False)
    if est is None:
        break
    errs.append((est[0] - (3 + f), est[1] - (-2 - f)))
if errs:
    errs = np.array(errs)
    plt.plot(fracs[:len(errs)], errs[:, 0], "o-", label="dy 误差")
    plt.plot(fracs[:len(errs)], errs[:, 1], "s-", label="dx 误差")
    plt.axhline(0, color="k", lw=0.5); plt.xlabel("真值的小数部分"); plt.ylabel("误差 (px)"); plt.legend(); plt.show()
    print("最大绝对误差", np.abs(errs).max().round(3), "px")

## 实验 2：周期图案的假峰

周期 8 px。平移 3 px 和平移 5 px 时，估计结果分别是多少？为什么 5 px 会出问题？

In [ ]:
for true in [(0, 3), (0, 5), (0, 9)]:
    est = attempt(mod.phase_correlation, periodic, np.roll(periodic, true, axis=(0, 1)), use_window=False)
    if est is not None:
        print(f"真值 {true} → 估计 ({est[0]:.2f}, {est[1]:.2f})")
print("对照第 16 章教程第 7 节：怎样用对位标记或限制搜索范围解决？")

## 实验 3：ZNCC 对亮度变化的不变性

In [ ]:
img = (ref - ref.min()) / np.ptp(ref) * 200
tpl = img[30:50, 40:64]
dim = img * 0.6 + 30
for name, target in [("原图", img), ("变暗加偏置", dim)]:
    m = attempt(mod.ncc_map, target, tpl)
    if m is None:
        break
    py, px = np.unravel_index(np.argmax(m), m.shape)
    print(f"{name}：峰值位置 ({py}, {px})，峰值 {m.max():.4f}")
if m is not None:
    plt.imshow(m, cmap="magma"); plt.colorbar(); plt.title("ZNCC 响应图"); plt.show()
if cv2 is not None:
    r = cv2.matchTemplate(img.astype(np.float32), tpl.astype(np.float32), cv2.TM_CCOEFF_NORMED)
    print("OpenCV TM_CCOEFF_NORMED 峰值位置", np.unravel_index(np.argmax(r), r.shape))

## 实验 4：噪声与加窗

真实图像边界不连续。下面用“裁剪”而不是循环平移制造位移，比较加窗和不加窗。

In [ ]:
big = texture((160, 160), seed=3)
rng = np.random.default_rng(1)
rows = []
for use_window in (False, True):
    e = []
    for _ in range(20):
        dy, dx = rng.integers(-6, 7, size=2)
        a = big[30:126, 30:126]
        b = big[30 - dy:126 - dy, 30 - dx:126 - dx] + rng.normal(scale=0.05 * big.std(), size=(96, 96))
        est = attempt(mod.phase_correlation, a, b, use_window=use_window)
        if est is None:
            break
        e.append(max(abs(est[0] - dy), abs(est[1] - dx)))
    if e:
        rows.append((use_window, np.mean(e), np.max(e)))
for w, mean_e, max_e in rows:
    print(f"加窗={w}：平均误差 {mean_e:.3f} px，最大 {max_e:.3f} px")

## 破坏实验与记录

- 把噪声加到纹理标准差的 50%，误差怎么变？
- 把纹理换成更平滑的（width=0.02），相位相关还可靠吗？
- 把实验 1 的曲线、实验 2 的失败案例写进 `实验记录.md`。